<a href="https://colab.research.google.com/github/dennisddschulz/cas-artificial-intelligence/blob/main/08_drl_einstieg/02_DRL_Policy_Improvement_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DRL Intro: MDP, Return, V/Q/Advantage – Monte-Carlo Evaluation (FrozenLake)

Ziel:
1) Episode sammeln (Rollout)
2) Returns G_t berechnen
3) Monte-Carlo Schätzung von V(s) und Q(s,a)
4) Advantage A(s,a) berechnen
5) Aus Q eine ε-greedy Policy ableiten
6) Zeigen, dass sich V(start) verbessert (Policy Improvement)


In [8]:
!pip -q install gymnasium

import gymnasium as gym
import numpy as np
from collections import defaultdict

SEED = 42
rng = np.random.default_rng(SEED)


In [9]:
import gymnasium as gym

env = gym.make("Taxi-v3")

s0, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n

print("nS:", nS, "nA:", nA, "start:", s0)

nS: 500 nA: 6 start: 386


## Policy und Rollout

- Policy: Funktion, die aus Zustand s eine Aktion a wählt.
- Rollout: wir lassen Agent+Env laufen und speichern (s, a, r) pro Schritt.


In [10]:
def random_policy(s, nA):
    return int(rng.integers(nA))

def rollout_episode(env, policy_fn, max_steps=200, seed=0):
    traj = []  # list of (s, a, r)
    s, _ = env.reset(seed=seed)
    for _ in range(max_steps):
        a = policy_fn(s, env.action_space.n)
        s2, r, terminated, truncated, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if terminated or truncated:
            break
    return traj

traj = rollout_episode(env, random_policy, seed=SEED)
print("episode length:", len(traj))
print("first steps:", traj[:8])


episode length: 200
first steps: [(386, 0, -1), (486, 4, -10), (486, 3, -1), (466, 2, -1), (486, 2, -1), (486, 5, -10), (486, 0, -1), (486, 4, -10)]


## Returns berechnen

Return G_t ist die discounted Summe der zukünftigen Rewards ab Schritt t.
Wir berechnen das rückwärts:
G = 0
G <- r + gamma * G


In [11]:
def compute_returns(traj, gamma=0.99):
    G = 0.0
    returns = []
    # HA - Warum reversed?
    for (s, a, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

gamma = 0.99
Gs = compute_returns(traj, gamma=gamma)
list(zip(traj[:8], Gs[:8]))


[((386, 0, -1), -378.5932359691751),
 ((486, 4, -10), -381.40730905977284),
 ((486, 3, -1), -375.1588980401746),
 ((466, 2, -1), -377.93828084866124),
 ((486, 2, -1), -380.74573823097097),
 ((486, 5, -10), -383.58155376865756),
 ((486, 0, -1), -377.3551048168258),
 ((486, 4, -10), -380.1566715321473)]

## MC Evaluation von V(s)

First-Visit MC:
- pro Episode zählt nur das erste Auftreten eines Zustands s
- V(s) = Durchschnitt der beobachteten Returns in s

```
seen = set()
for t, (s, a, r) in enumerate(traj):
    if s in seen:
        continue
    seen.add(s)
    V[s] += G[t]
```

Every-Visit MC:
- pro Episode zählt jedes Auftreten eines Zustands s
- V(s) ist der Durchschnitt der beobachteten Return über alle Besuche von s in allen Episoden.
```
for t, (s, a, r) in enumerate(traj):
    V[s] += G[t]
```


In [12]:
def mc_evaluate_V(env, policy_fn, episodes=3000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        #seen = set()
        for t, (s, a, r) in enumerate(traj):
            #if s in seen:
            #    continue
            #seen.add(s)
            returns_sum[s] += Gs[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V

V_rand = mc_evaluate_V(env, random_policy, episodes=4000, gamma=gamma, seed=SEED)

s0, _ = env.reset(seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))


V_random(start): -227.8028


In [13]:
V_rand

{386: -227.80277618054404,
 286: -232.07842638901047,
 186: -228.72880248013496,
 86: -205.78517239387344,
 98: -191.86025496541038,
 198: -192.07633144153527,
 298: -194.80508912405043,
 398: -200.73328069555512,
 178: -192.55746992388313,
 158: -190.17064760430412,
 58: -188.037279803961,
 78: -187.22997282330135,
 258: -174.63829845895035,
 238: -161.64704857622178,
 218: -137.23041457938237,
 318: -97.44638717266005,
 418: -56.958040362683455,
 118: -162.59965995557675,
 18: -177.1651504466479,
 2: -202.29373800634724,
 22: -213.13279353392397,
 102: -209.84742497240322,
 202: -223.85946764692127,
 222: -226.2383167196556,
 242: -228.36707228096637,
 262: -233.06714332214574,
 362: -238.9115188437634,
 462: -245.51566847707664,
 382: -239.73211721317216,
 482: -242.18295080846048,
 282: -241.03805020093031,
 162: -233.25103957907305,
 142: -233.87476965671704,
 122: -215.97253703605742,
 322: -229.87378864859895,
 342: -224.9285770146329,
 442: -231.2770693312455,
 422: -226.090221

## MC Evaluation von Q(s,a)

Analog:
- wir mitteln Returns pro (s,a)
- daraus können wir greedy / ε-greedy Policies bauen


In [14]:
def mc_evaluate_Q(env, policy_fn, episodes=6000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen_sa = set()
        for t, (s, a, r) in enumerate(traj):
            key = (s, a)
            if key in seen_sa:
                continue
            seen_sa.add(key)
            returns_sum[key] += Gs[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_count}
    return Q

Q_rand = mc_evaluate_Q(env, random_policy, episodes=8000, gamma=gamma, seed=SEED)
print("Q entries:", list(Q_rand.items())[:5])


Q entries: [((386, 3), -245.63050195550255), ((366, 4), -254.21995512338552), ((366, 2), -244.51968950793648), ((366, 5), -253.28715608360596), ((366, 1), -252.87142613733107)]


## Advantage A(s,a)

A(s,a) = Q(s,a) - V(s)
Interpretation: wie viel besser/schlechter ist Aktion a gegenüber dem "Durchschnitt" in s.


In [15]:
def advantage(V, Q, s, a):
    return Q.get((s, a), 0.0) - V.get(s, 0.0)

# Advantage im Startzustand für alle Aktionen
adv_start = [(a, advantage(V_rand, Q_rand, s0, a)) for a in range(nA)]
adv_start


[(0, -20.093831233853194),
 (1, -19.76179395089474),
 (2, -20.49953894460515),
 (3, -17.82772577495851),
 (4, -21.696980279276715),
 (5, -31.9976912893695)]

## Policy Improvement: ε-greedy aus Q

- greedy: a = argmax_a Q(s,a)
- ε-greedy: mit Wahrscheinlichkeit ε zufällig, sonst greedy

Dann evaluieren wir die neue Policy wieder mit MC und vergleichen V(start).


In [16]:
def epsilon_greedy_policy_from_Q(Q, nA, eps=0.1):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        qs = [Q.get((s, a), 0.0) for a in range(nA)]
        return int(np.argmax(qs))
    return policy

pi_eps = epsilon_greedy_policy_from_Q(Q_rand, nA, eps=0.1)

V_eps = mc_evaluate_V(env, pi_eps, episodes=4000, gamma=gamma, seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))
print("V_eps(start):   ", round(V_eps.get(s0, 0.0), 4))


V_random(start): -227.8028
V_eps(start):    -76.8201


## Mini Loop: wiederholte Verbesserung (Iteration)

Wir wiederholen:
1) Q unter aktueller Policy schätzen
2) neue ε-greedy Policy bauen
3) V(start) loggen

Achtung: Das ist noch nicht "Policy Iteration" im strengen Sinn,
aber zeigt sehr gut die Grundidee: Bessere Wertschätzungen (V/Q) → bessere Entscheidungsgrundlage → verbesserte Policy


In [17]:
def policy_improvement_loop(env, init_policy, iters=5, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=0.99, seed=0):
    policy = init_policy
    history = []

    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)
        policy = epsilon_greedy_policy_from_Q(Q, env.action_space.n, eps=eps)
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        history.append((k, V.get(s0, 0.0)))

    return history

hist = policy_improvement_loop(env, random_policy, iters=50, eps=0.1, episodes_Q=100000, episodes_V=5000, gamma=gamma, seed=SEED)
hist


[(0, -108.04006679101478),
 (1, -87.76930573910306),
 (2, -183.08503753378753),
 (3, -370.33906234560624),
 (4, -429.0179447711177),
 (5, -494.82517078030025)]

## Was haben wir heute gelernt?

- Reward vs Return: Return ist das Ziel, nicht der einzelne Reward.
- V(s) und Q(s,a) sind Erwartungswerte von Returns.
- Monte-Carlo schätzt diese Werte aus Episoden (ohne Modell von P).
- Advantage erklärt "wie gut ist diese Aktion relativ zum Durchschnitt in s".
- Aus Q kann man eine bessere Policy ableiten (ε-greedy).


## Policy Improvement - Wie lernt der Agent?

In [18]:
def epsilon_linear(k, eps0=0.3, eps_min=0.02, decay_steps=10):
    # fällt linear ab von eps0 zu eps_min über decay_steps Iterationen
    eps = eps0 - (eps0 - eps_min) * (k / decay_steps)
    return float(max(eps_min, eps))

def epsilon_exp(k, eps0=0.3, eps_min=0.02, alpha=0.85):
    eps = eps0 * (alpha ** k)
    return float(max(eps_min, eps))


In [19]:
def greedy_action_from_Q(Q, s, nA):
    qs = [Q.get((s, a), 0.0) for a in range(nA)]
    return int(np.argmax(qs))

def make_epsilon_greedy_policy(Q, nA, eps):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        return greedy_action_from_Q(Q, s, nA)
    return policy


In [20]:
def evaluate_success_rate(env, policy_fn, episodes=1000, seed=0, max_steps=200):
    successes = 0
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        total_reward = 0
        for _ in range(max_steps):
            a = policy_fn(s, env.action_space.n)
            s, r, terminated, truncated, _ = env.step(a)
            total_reward += r
            if terminated or truncated:
                break
        if total_reward > 0:
            successes += 1
    return successes / episodes


In [26]:
def policy_improvement_loop_mc_control_decay(
    env,
    init_policy,
    iters=50,
    eps_schedule="linear",      # "linear" oder "exp"
    eps0=0.5,
    eps_min=0.1,
    decay_steps=18,             # für linear
    alpha=0.85,                 # für exp
    episodes_Q=100000,
    episodes_V=5000,
    eval_episodes=1000,
    gamma=0.99,
    seed=0
):
    policy = init_policy
    history = []
    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        # 1) Evaluate current policy -> estimate Q
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)

        # 2) Choose epsilon for this iteration
        if eps_schedule == "linear":
            eps = epsilon_linear(k, eps0=eps0, eps_min=eps_min, decay_steps=decay_steps)
        elif eps_schedule == "exp":
            eps = epsilon_exp(k, eps0=eps0, eps_min=eps_min, alpha=alpha)
        else:
            raise ValueError("eps_schedule must be 'linear' or 'exp'")

        # 3) Improve policy using greedy(max) with epsilon exploration
        policy = make_epsilon_greedy_policy(Q, env.action_space.n, eps=eps)

        # 4) Track progress
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        v0 = float(V.get(s0, 0.0))
        sr = evaluate_success_rate(env, policy, episodes=eval_episodes, seed=seed + 3000*k)

        history.append({"iter": k, "eps": eps, "V(start)": v0, "success_rate": sr})
        print(f"iter={k:02d}  eps={eps:.3f}  V(start)={v0:.4f}  success_rate={sr:.3f}")

    return policy, Q, history


In [27]:
trained_epsilon_greedy_policy_func, q_final, hist = policy_improvement_loop_mc_control_decay(env, random_policy, seed=SEED)

iter=00  eps=0.700  V(start)=-162.8965  success_rate=0.004
iter=01  eps=0.683  V(start)=-89.7783  success_rate=0.006
iter=02  eps=0.665  V(start)=-92.6344  success_rate=0.008
iter=03  eps=0.648  V(start)=-74.1513  success_rate=0.008
iter=04  eps=0.631  V(start)=-74.0947  success_rate=0.014
iter=05  eps=0.614  V(start)=-71.3710  success_rate=0.011
iter=06  eps=0.596  V(start)=-68.4803  success_rate=0.014
iter=07  eps=0.579  V(start)=-65.0623  success_rate=0.027
iter=08  eps=0.562  V(start)=-56.8734  success_rate=0.019
iter=09  eps=0.545  V(start)=-57.4971  success_rate=0.032
iter=10  eps=0.527  V(start)=-46.0115  success_rate=0.028
iter=11  eps=0.510  V(start)=-47.8218  success_rate=0.032
iter=12  eps=0.493  V(start)=-46.2221  success_rate=0.041
iter=13  eps=0.476  V(start)=-41.9880  success_rate=0.044
iter=14  eps=0.459  V(start)=-34.7157  success_rate=0.047
iter=15  eps=0.441  V(start)=-32.3887  success_rate=0.059
iter=16  eps=0.424  V(start)=-33.6200  success_rate=0.076
iter=17  eps=

In [23]:
q_final

{(169, 1): -309.167679077196,
 (69, 3): -285.55164536808564,
 (49, 0): -291.99825203000444,
 (149, 0): -293.97756697625914,
 (249, 3): -294.4912355087045,
 (229, 3): -303.00792970656016,
 (209, 2): -306.421614011616,
 (229, 0): -438.8297718518547,
 (329, 0): -477.3591009510573,
 (429, 5): -499.1890554178716,
 (429, 1): -419.68235067478696,
 (429, 4): -426.53507774280354,
 (429, 3): -435.1032247935868,
 (429, 0): -434.53050468140094,
 (429, 2): -343.41672569409616,
 (449, 2): -355.02820251587417,
 (449, 0): -302.12334064086286,
 (449, 1): -362.3025164590407,
 (349, 3): -413.08035861636665,
 (449, 3): -396.28696153697206,
 (473, 0): -210.3329533853767,
 (473, 1): -195.72998622054257,
 (373, 0): -204.08922024363716,
 (473, 5): -206.17594297848905,
 (473, 3): -197.96348749026754,
 (473, 2): -358.14542538557146,
 (493, 5): -398.6228147470491,
 (493, 0): -358.2790800173991,
 (493, 2): -364.5109540810583,
 (493, 4): -363.362105839966,
 (493, 3): -190.91529156238684,
 (412, 2): -623.0377546400

In [24]:

greedy_trained_policy = make_epsilon_greedy_policy(q_final, env.action_space.n, eps=0.0)

In [25]:
_, q_evolution, hist = policy_improvement_loop_mc_control_decay(env, greedy_trained_policy, seed=SEED)

iter=00  eps=0.300  V(start)=-112.2451  success_rate=0.011
iter=01  eps=0.272  V(start)=-337.9059  success_rate=0.000
iter=02  eps=0.244  V(start)=-220.3101  success_rate=0.005
iter=03  eps=0.216  V(start)=-247.1520  success_rate=0.010
iter=04  eps=0.188  V(start)=-243.2160  success_rate=0.009
iter=05  eps=0.160  V(start)=-372.3845  success_rate=0.007
iter=06  eps=0.132  V(start)=-368.3860  success_rate=0.015
iter=07  eps=0.104  V(start)=-446.8034  success_rate=0.024
